In [19]:
# notebooks/01_reproduce_champion.ipynb
import sys
import os
import pandas as pd
import numpy as np

# Point to your custom source directory
sys.path.append(os.path.abspath('../'))
from src.data_loader import load_validation_splits
from src.metrics import calculate_gini_and_ks

# 2. Ingest the data directly from disk
df_train_tweaked, df_val_final, df_oot_final, df_scorecard_points = load_validation_splits()
print(f"✅ Data successfully synchronized from disk storage pool.")
print(f"Loaded Train Shape: {df_train_tweaked.shape} | OOT Shape: {df_oot_final.shape}")


# 1. HARDCODE EXPERT CONFIGURATION FROM CHAMPION BLUEPRINT
CHAMPION_METADATA = {
    "Scaling_Parameters": {
        "target_score": 600,
        "target_odds": 5.0,
        "pdo": 20,
    },
    "Features": [
        'avg_cur_bal_woe', 'tot_hi_cred_lim_woe', 'tot_cur_bal_woe', 
        'percent_bc_gt_75_woe', 'mo_sin_old_rev_tl_op_woe', 'loan_amnt_woe',
        'fico_score_woe', 'dti_woe', 'annual_inc_woe', 'acc_open_past_24mths_woe', 
        'mo_sin_rcnt_tl_woe', 'inq_last_6mths_woe', 'mths_since_recent_inq_woe', 
        'mort_acc_woe', 'bc_open_to_buy_woe', 'total_rev_hi_lim_woe', 
        'mths_since_recent_bc_woe', 'grade_woe', 'term_woe', 'verification_status_woe'
    ]
}

# 2. MATCH REPLICATION TEST CORES
print("Replication Engine Initialized.")
print(f"Targeting verification of {len(CHAMPION_METADATA['Features'])} scorecard attributes.")


# notebooks/01_reproduce_champion.ipynb (CORRECTED FEATURE ISOLATION)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

print("\n=== RUNNING CHAMPION PERFORMANCE REPLICATION ===")

# FIX: Derive the final features directly from your scorecard points table instead of raw dataframe columns
final_features = [f"{col}_woe" for col in df_scorecard_points['Variable'].unique()]

# Remove the two problematic features dropped during statsmodels validation step
problematic_drops = ['mo_sin_rcnt_rev_tl_op_woe', 'num_tl_op_past_12m_woe']
final_features = [col for col in final_features if col not in problematic_drops]

# Remove suffixes to know what raw variables we need to map for validation/OOT
numerical_originals = [col.replace('_woe', '') for col in final_features if not col.startswith(('grade', 'term', 'verification_status'))]
categorical_originals = ['grade', 'term', 'verification_status']

# 2. Re-create the WoE columns for Validation and OOT sets using your scorecard points lookup rules
print("Transforming validation and OOT matrices using scorecard points mappings...")
for col in numerical_originals:
    pts_sub = df_scorecard_points[df_scorecard_points['Variable'] == col]
    woe_lookup = pts_sub.set_index('Bin/Category')['WoE'].to_dict()
    
    for bin_str, woe_val in woe_lookup.items():
        raw_left = bin_str.split(',')[0].replace('(', '').replace('[', '').strip()
        raw_right = bin_str.split(',')[1].replace(')', '').replace(']', '').strip()
        
        left_side = float('-inf') if raw_left == '-inf' else float(raw_left)
        right_side = float('inf') if raw_right == 'inf' else float(raw_right)
        
        if bin_str.startswith('(') and bin_str.endswith(']'):
            df_val_final.loc[(df_val_final[col] > left_side) & (df_val_final[col] <= right_side), f'{col}_woe'] = woe_val
            df_oot_final.loc[(df_oot_final[col] > left_side) & (df_oot_final[col] <= right_side), f'{col}_woe'] = woe_val
        else:
            df_val_final.loc[(df_val_final[col] >= left_side) & (df_val_final[col] <= right_side), f'{col}_woe'] = woe_val
            df_oot_final.loc[(df_oot_final[col] >= left_side) & (df_oot_final[col] <= right_side), f'{col}_woe'] = woe_val

for col in categorical_originals:
    pts_sub = df_scorecard_points[df_scorecard_points['Variable'] == col]
    cat_to_woe = pts_sub.set_index('Bin/Category')['WoE'].to_dict()
    
    df_val_final[f'{col}_woe'] = df_val_final[col].fillna('Missing').astype(str).map(cat_to_woe)
    df_oot_final[f'{col}_woe'] = df_oot_final[col].fillna('Missing').astype(str).map(cat_to_woe)

# Fill any structural missing values with 0 to maintain regression stability
X_train_rep = df_train_tweaked[final_features].apply(pd.to_numeric, errors='coerce').fillna(0)
y_train_rep = df_train_tweaked['bad'].astype(int)

X_val_rep = df_val_final[final_features].apply(pd.to_numeric, errors='coerce').fillna(0)
y_val_rep = df_val_final['bad'].astype(int)

X_oot_rep = df_oot_final[final_features].apply(pd.to_numeric, errors='coerce').fillna(0)
y_oot_rep = df_oot_final['bad'].astype(int)

# 3. Train production clone
lr_clone = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
lr_clone.fit(X_train_rep, y_train_rep)

# 4. Run calculations using your official src/metrics.py engine
def verify_metric_sync(X, y, dataset_name):
    probs = lr_clone.predict_proba(X)[:, 1]
    metrics = calculate_gini_and_ks(y, probs)
    print(f"📈 [{dataset_name}] Replicated AUC: {metrics['AUC']:.4f} | Gini: {metrics['Gini']:.4f} | KS: {metrics['KS']:.4f}")
    return metrics

train_metrics = verify_metric_sync(X_train_rep, y_train_rep, "Train Set")
val_metrics = verify_metric_sync(X_val_rep, y_val_rep, "In-Time Validation Set")
oot_metrics = verify_metric_sync(X_oot_rep, y_oot_rep, "Out-of-Time Set")

✅ Data successfully synchronized from disk storage pool.
Loaded Train Shape: (708853, 65) | OOT Shape: (86951, 30)
Replication Engine Initialized.
Targeting verification of 20 scorecard attributes.

=== RUNNING CHAMPION PERFORMANCE REPLICATION ===
Transforming validation and OOT matrices using scorecard points mappings...
📈 [Train Set] Replicated AUC: 0.7193 | Gini: 0.4386 | KS: 0.3166
📈 [In-Time Validation Set] Replicated AUC: 0.7213 | Gini: 0.4427 | KS: 0.3217
📈 [Out-of-Time Set] Replicated AUC: 0.6998 | Gini: 0.3996 | KS: 0.2882
